In [1]:
import os
import re
import requests
import pandas as pd
from collections import defaultdict
from urllib.parse import urljoin
from bs4 import BeautifulSoup

# 1. fonts.html → df
with open('./fonts/fonts.html', encoding='utf-8') as f:
    html = f.read()

soup = BeautifulSoup(html, 'html.parser')

rows = []
for card in soup.select('div.font-card[data-font]'):
    name = card.select_one('.font-name')
    meta = card.select_one('.font-meta')
    rows.append({
        'font-name': name.get_text(strip=True) if name else None,
        'font-family': card.get('data-font'),
        'meta': meta.get_text(' ', strip=True) if meta else None,
    })
df = pd.DataFrame(rows)

# 2. fonts.css 파싱
with open('./fonts/fonts.css', encoding='utf-8') as f:
    css = f.read()

FACE_RE = re.compile(r"@font-face\s*\{([^}]*)\}", re.DOTALL)

faces = defaultdict(lambda: {'font-weight': [], 'src': []})
for block in FACE_RE.findall(css):
    fam = re.search(r"font-family:\s*['\"]?([^'\";]+)", block)
    url = re.search(r"url\(['\"]?([^'\")]+)", block)
    wt  = re.search(r"font-weight:\s*([^;]+)", block)
    if not fam:
        continue
    key = fam.group(1).strip()
    faces[key]['font-weight'].append(wt.group(1).strip() if wt else None)
    faces[key]['src'].append(url.group(1).strip() if url else None)

# 3. weight 기준 정렬
def sort_key(w):
    if w is None:
        return 9999
    m = re.search(r'\d+', str(w))
    return int(m.group()) if m else 9999

for v in faces.values():
    pairs = sorted(zip(v['font-weight'], v['src']), key=lambda x: sort_key(x[0]))
    v['font-weight'] = [p[0] for p in pairs]
    v['src'] = [p[1] for p in pairs]

# 4. merge
faces_df = pd.DataFrame([{'font-family': k, **v} for k, v in faces.items()])
df = df.merge(faces_df, on='font-family', how='left')

print(df.head())

# 5. download
BASE = 'https://typedia.kr/'
OUT_DIR = './fonts/woff'
os.makedirs(OUT_DIR, exist_ok=True)

session = requests.Session()
session.headers.update({'User-Agent': 'Mozilla/5.0'})

def safe_name(s):
    return re.sub(r'[^\w\-]+', '_', str(s)).strip('_')

def weight_label(wt):
    if not wt:
        return 'x'
    nums = re.findall(r'\d+', str(wt))
    if len(nums) >= 2:
        return f'VF{nums[0]}-{nums[-1]}'
    if len(nums) == 1:
        return nums[0]
    return safe_name(wt)

for _, row in df.iterrows():
    family = row['font-family']
    srcs = row.get('src') or []
    weights = row.get('font-weight') or []
    if not isinstance(srcs, list):
        continue

    for src, wt in zip(srcs, weights):
        if not src:
            continue

        url = src if src.startswith('http') else urljoin(BASE, src.lstrip('/'))
        ext = os.path.splitext(src.split('?')[0])[1] or '.woff'

        fname = f"{safe_name(family)}-{weight_label(wt)}"
        out_path = os.path.join(OUT_DIR, fname + ext)

        if os.path.exists(out_path):
            print('skip', out_path)
            continue

        try:
            r = session.get(url, timeout=30)
            r.raise_for_status()
            with open(out_path, 'wb') as f:
                f.write(r.content)
            print('saved', out_path)
        except Exception as e:
            print('fail', url, e)

  font-name        font-family         meta font-weight  \
0   (구)코레일체             KORAIL  한국철도공사 / 기관       [400]   
1   [김씨]손글씨              KJDHW     김정대 / 개인       [400]   
2   116수박화체      116watermelon    CARA / 개인       [400]   
3   116앵덕정직  116angduk_honesty    CARA / 개인       [400]   
4   116앵무부리       116angmuburi    CARA / 개인       [400]   

                                              src  
0                [/wp-content/fonts/KORAIL.woff2]  
1                 [/wp-content/fonts/KJDHW.woff2]  
2         [/wp-content/fonts/116watermelon.woff2]  
3  [/wp-content/fonts/116angduk_honesty1.5.woff2]  
4          [/wp-content/fonts/116angmuburi.woff2]  
saved ./fonts/woff/KORAIL-400.woff2
saved ./fonts/woff/KJDHW-400.woff2
saved ./fonts/woff/116watermelon-400.woff2
saved ./fonts/woff/116angduk_honesty-400.woff2
saved ./fonts/woff/116angmuburi-400.woff2
saved ./fonts/woff/11STREET_Gothic-300.woff2
saved ./fonts/woff/11STREET_Gothic-400.woff2
saved ./fonts/woff/11STREET_Gothic-700.w

In [1]:
import os
import re
import io
import shutil
import logging
import subprocess
from collections import Counter
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
from fontTools.ttLib import TTFont, newTable
from fontTools.pens.ttGlyphPen import TTGlyphPen
from fontTools.pens.cu2quPen import Cu2QuPen
from fontTools.varLib.instancer import instantiateVariableFont

SRC_DIR = './fonts/woff'
OUT_DIR = './fonts/ttf'
WORKERS = 32
os.makedirs(OUT_DIR, exist_ok=True)

OTS = shutil.which('ots-sanitize')

MAX_ERR = 0.5
POST_FORMAT = 2.0
REVERSE_DIRECTION = True

def glyphs_to_quadratic(glyphs, max_err=MAX_ERR, reverse_direction=REVERSE_DIRECTION):
    quad = {}
    for name, glyph in glyphs.items():
        pen = TTGlyphPen(glyphs)
        cu2qu = Cu2QuPen(pen, max_err, reverse_direction=reverse_direction)
        glyph.draw(cu2qu)
        quad[name] = pen.glyph()
    return quad

def convert_cff_to_glyf(font):
    assert 'CFF ' in font or 'CFF2' in font
    glyph_order = font.getGlyphOrder()
    glyph_set = font.getGlyphSet()
    glyphs = {g: glyph_set[g] for g in glyph_order}
    quad = glyphs_to_quadratic(glyphs)

    if 'CFF ' in font:
        del font['CFF ']
    if 'CFF2' in font:
        del font['CFF2']
    if 'VORG' in font:
        del font['VORG']

    glyf = newTable('glyf')
    glyf.glyphOrder = glyph_order
    glyf.glyphs = quad
    font['glyf'] = glyf

    loca = newTable('loca')
    font['loca'] = loca

    font.sfntVersion = '\x00\x01\x00\x00'

    if 'post' in font:
        post = font['post']
        post.formatType = POST_FORMAT
        if not hasattr(post, 'extraNames'):
            post.extraNames = []
        if not hasattr(post, 'mapping'):
            post.mapping = {}
    return font

def is_variable(font):
    return 'fvar' in font

def is_cff(font):
    return 'CFF ' in font or 'CFF2' in font

def to_glyf(font):
    if is_cff(font):
        convert_cff_to_glyf(font)
    return font

def prune_empty_cmap(font):
    if 'cmap' in font:
        cmap = font['cmap']
        cmap.tables = [t for t in cmap.tables if getattr(t, 'cmap', None)]

def instantiate_all_axes(src, w):
    vf = TTFont(src)
    vf.flavor = None
    axes = {a.axisTag: a.defaultValue for a in vf['fvar'].axes}
    axes['wght'] = w
    return instantiateVariableFont(vf, axes)

def validate(path):
    try:
        f = TTFont(path)
        for tag in ('cmap', 'head', 'hhea', 'hmtx', 'maxp', 'name', 'post', 'glyf', 'loca'):
            if tag not in f:
                return False, f'missing {tag}'
        for tag in f.keys():
            _ = f[tag]
        glyf = f['glyf']
        for name in f.getGlyphOrder():
            g = glyf[name]
            if hasattr(g, 'numberOfContours') and g.numberOfContours > 0:
                _ = g.coordinates
        best = f['cmap'].getBestCmap()
        if not best:
            return False, 'empty cmap'
        if not f['hmtx'].metrics:
            return False, 'empty hmtx'
        buf = io.BytesIO()
        f.save(buf)
        f.close()
    except Exception as e:
        return False, f'fonttools: {e}'

    if OTS:
        try:
            r = subprocess.run(
                [OTS, path],
                capture_output=True, text=True, timeout=30
            )
            if r.returncode != 0:
                msg = (r.stderr or r.stdout).strip().splitlines()
                return False, 'ots: ' + (msg[-1] if msg else 'failed')
            err_lines = [l for l in (r.stderr or '').splitlines() if 'ERROR' in l]
            if err_lines:
                return False, 'ots: ' + err_lines[0]
        except subprocess.TimeoutExpired:
            return False, 'ots: timeout'
        except Exception as e:
            return False, f'ots: {e}'
    else:
        return True, 'ok (ots missing)'

    return True, 'ok'

def process_one(fname):
    logging.getLogger('fontTools').setLevel(logging.ERROR)

    out_results = []
    src = os.path.join(SRC_DIR, fname)
    if not os.path.isfile(src):
        return out_results
    stem, ext = os.path.splitext(fname)
    ext = ext.lower()
    if ext not in ('.woff', '.woff2', '.ttf', '.otf'):
        return out_results

    try:
        font = TTFont(src)
        font.flavor = None
    except Exception as e:
        return [(fname, 'load_fail', str(e))]

    try:
        if is_variable(font):
            fvar = font['fvar']
            wght = next((a for a in fvar.axes if a.axisTag == 'wght'), None)
            base_stem = re.sub(r'-VF\d+-\d+$', '', stem)

            if wght is None:
                font = to_glyf(font)
                font.flavor = None
                prune_empty_cmap(font)
                out = os.path.join(OUT_DIR, f'{base_stem}.ttf')
                font.save(out)
                ok, msg = validate(out)
                out_results.append((os.path.basename(out), 'ok' if ok else 'invalid', msg))
                return out_results

            lo = int(round(wght.minValue / 100.0)) * 100
            hi = int(round(wght.maxValue / 100.0)) * 100

            for w in range(lo, hi + 1, 100):
                if w < wght.minValue or w > wght.maxValue:
                    continue
                try:
                    inst = instantiate_all_axes(src, w)
                    inst = to_glyf(inst)
                    inst.flavor = None
                    prune_empty_cmap(inst)
                    out = os.path.join(OUT_DIR, f'{base_stem}-{w}.ttf')
                    inst.save(out)
                    ok, msg = validate(out)
                    out_results.append((os.path.basename(out), 'ok' if ok else 'invalid', msg))
                except Exception as e:
                    out_results.append((f'{base_stem}-{w}.ttf', 'instancing_fail', str(e)))
        else:
            font = to_glyf(font)
            font.flavor = None
            prune_empty_cmap(font)
            out = os.path.join(OUT_DIR, f'{stem}.ttf')
            font.save(out)
            ok, msg = validate(out)
            out_results.append((os.path.basename(out), 'ok' if ok else 'invalid', msg))
    except Exception as e:
        out_results.append((fname, 'convert_fail', str(e)))

    return out_results

if __name__ == '__main__':
    logging.getLogger('fontTools').setLevel(logging.ERROR)

    files = sorted(os.listdir(SRC_DIR))
    results = []

    with ProcessPoolExecutor(max_workers=WORKERS) as ex:
        futures = {ex.submit(process_one, fn): fn for fn in files}
        for fut in tqdm(as_completed(futures), total=len(futures)):
            fn = futures[fut]
            try:
                res = fut.result()
            except Exception as e:
                print('worker fail', fn, e)
                results.append((fn, 'worker_fail', str(e)))
                continue
            for r in res:
                results.append(r)
                if r[1] != 'ok':
                    print(r[1], r[0], r[2])

    summary = Counter(r[1] for r in results)
    print('\n=== summary ===')
    print(summary)

    fails = [r for r in results if r[1] != 'ok']
    if fails:
        print('\n=== failures ===')
        for r in fails:
            print(r)

100%|██████████| 1143/1143 [01:53<00:00, 10.05it/s]


=== summary ===
Counter({'ok': 1237})


In [1]:
import os
import re
import glob

FONT_DIR = "./fonts/ttf"

all_ttf = glob.glob(os.path.join(FONT_DIR, "*.ttf"))
pattern = re.compile(r"-\d+\.ttf$")

non_standard = [os.path.basename(p) for p in all_ttf if not pattern.search(p)]
non_standard.sort()

print(f"전체: {len(all_ttf)}개 / 비표준: {len(non_standard)}개")
for f in non_standard:
    print(f"  {f}")

전체: 1237개 / 비표준: 1개
  ClimateCrisis-x.ttf


In [3]:
import os
import re
import glob
import io
from fontTools.ttLib import TTFont
from fontTools.varLib.instancer import instantiateVariableFont

FONT_DIR = "./fonts/ttf"
pattern = re.compile(r"-\d+\.ttf$")

non_standard = sorted([
    os.path.join(FONT_DIR, os.path.basename(p))
    for p in glob.glob(os.path.join(FONT_DIR, "*.ttf"))
    if not pattern.search(p)
])
print(f"비표준 파일: {len(non_standard)}개\n")

for path in non_standard:
    fname = os.path.basename(path)
    stem = os.path.splitext(fname)[0]

    try:
        font = TTFont(path)
    except Exception as e:
        print(f"❌ {fname} — 로드 실패: {e}")
        continue

    has_fvar = "fvar" in font
    wght_axis = None
    if has_fvar:
        wght_axis = next((a for a in font["fvar"].axes if a.axisTag == "wght"), None)

    if wght_axis:
        # variable font + wght axis → 100~900 instancing
        lo = int(round(wght_axis.minValue / 100.0)) * 100
        hi = int(round(wght_axis.maxValue / 100.0)) * 100
        font.close()

        created = []
        for w in range(max(lo, 100), min(hi, 900) + 1, 100):
            if w < wght_axis.minValue or w > wght_axis.maxValue:
                continue
            out_name = f"{stem}-{w}.ttf"
            out_path = os.path.join(FONT_DIR, out_name)
            if os.path.exists(out_path):
                created.append(f"{w}(존재)")
                continue
            try:
                vf = TTFont(path)
                axes = {a.axisTag: a.defaultValue for a in vf["fvar"].axes}
                axes["wght"] = w
                inst = instantiateVariableFont(vf, axes)
                inst.flavor = None
                inst.save(out_path)
                inst.close()
                vf.close()
                created.append(str(w))
            except Exception as e:
                created.append(f"{w}(실패:{e})")

        print(f"✅ {fname} → VF instancing: [{', '.join(created)}]")
        os.remove(path)
        print(f"   🗑️ 원본 삭제")

    else:
        # non-variable 또는 wght 없음 → OS/2 weight로 rename
        weight_class = font["OS/2"].usWeightClass if "OS/2" in font else None
        font.close()

        if weight_class:
            w = int(round(weight_class / 100.0)) * 100
            w = max(100, min(900, w))
        else:
            w = 400

        new_name = f"{stem}-{w}.ttf"
        new_path = os.path.join(FONT_DIR, new_name)

        if os.path.exists(new_path):
            print(f"⚠️ {fname} → {new_name} (이미 존재, skip)")
        else:
            os.rename(path, new_path)
            print(f"✅ {fname} → {new_name} (OS/2: {weight_class})")

비표준 파일: 1개

✅ ClimateCrisis-x.ttf → ClimateCrisis-x-400.ttf (OS/2: 400)
